# Parallel Experiments with Persistent Disks

This notebook demonstrates:
1. Creating a base environment on a persistent disk
2. Making changes (installing packages, modifying code)
3. Cloning the disk for parallel experiments
4. Running two experiments simultaneously on different GPUs
5. Comparing results and measuring timings

In [ ]:
%pip install -e .. -q

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from gpu_dev import GpuDev

client = GpuDev()
print(f"SDK v{__import__('gpu_dev').__version__}")

## Step 1: Create Base Environment on Persistent Disk

Reserve a GPU with a persistent disk and set up the base experiment.

In [ ]:
t0 = time.time()

base = client.reserve(
    gpu_type="t4",
    gpu_count=1,
    hours=1,
    disk_name="experiment-base",
    name="base-setup",
)

reserve_time = time.time() - t0
print(f"Reserved in {reserve_time:.1f}s")
print(f"Disk: {base.disk_name}")
print(f"GPU:  {base.gpu_type} x{base.gpu_count}")

In [ ]:
# Set up the base experiment: install packages + write training script
base.exec("pip install -q wandb timm")

# Write a parameterized training script
base.exec(r"""
cat > /home/dev/train.py << 'SCRIPT'
import torch
import torch.nn as nn
import time
import json
import os
import sys

# Read experiment config from env
LR = float(os.environ.get('LR', '0.001'))
BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '64'))
EPOCHS = int(os.environ.get('EPOCHS', '5'))
EXP_NAME = os.environ.get('EXP_NAME', 'default')

print(f"Experiment: {EXP_NAME}")
print(f"Config: lr={LR}, batch_size={BATCH_SIZE}, epochs={EPOCHS}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

# Simple CNN on synthetic data
model = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64 * 8 * 8, 10)
).cuda()

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

results = {'experiment': EXP_NAME, 'lr': LR, 'batch_size': BATCH_SIZE, 'losses': [], 'epoch_times': []}

for epoch in range(EPOCHS):
    t_start = time.time()
    epoch_loss = 0
    for step in range(50):
        x = torch.randn(BATCH_SIZE, 3, 32, 32, device='cuda')
        y = torch.randint(0, 10, (BATCH_SIZE,), device='cuda')
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / 50
    epoch_time = time.time() - t_start
    results['losses'].append(avg_loss)
    results['epoch_times'].append(epoch_time)
    print(f"  Epoch {epoch+1}/{EPOCHS}: loss={avg_loss:.4f} ({epoch_time:.2f}s)")

results['final_loss'] = results['losses'][-1]
results['avg_epoch_time'] = sum(results['epoch_times']) / len(results['epoch_times'])

with open(f'/home/dev/results_{EXP_NAME}.json', 'w') as f:
    json.dump(results, f)
print(f"Results saved to /home/dev/results_{EXP_NAME}.json")
SCRIPT
""")

# Verify
result = base.exec("ls -la /home/dev/train.py && python3 -c 'import wandb, timm; print(\"packages OK\")'")
print(result.stdout.strip())

In [ ]:
## Step 2: Clone the Disk (Live)

Clone the disk while the base reservation is still running.
The SDK snapshots the live volume and creates a new disk from it — no need to cancel first.

In [ ]:
# Clone the disk for the second experiment
t0 = time.time()
client.clone_disk("experiment-base", "experiment-variant")
clone_time = time.time() - t0
print(f"Disk cloned in {clone_time:.1f}s")

# Show both disks
for disk in client.disks():
    if 'experiment' in disk.name:
        print(f"  {disk.name:25s}  {disk.size_gb}GB  {disk.snapshot_count} snapshots")

## Step 3: Run Parallel Experiments

Cancel the base reservation now (we have the clone), then launch two
reservations simultaneously — one on the original disk (high LR),
one on the cloned disk (low LR).

In [ ]:
# Now cancel the base — we have the clone, both disks are independent
base.cancel()
print(f"Base cancelled — both disks ready for parallel experiments")

In [ ]:
experiments = [
    {"name": "high-lr",  "disk": "experiment-base",    "env": "LR=0.01 BATCH_SIZE=128 EPOCHS=5 EXP_NAME=high_lr"},
    {"name": "low-lr",   "disk": "experiment-variant",  "env": "LR=0.0001 BATCH_SIZE=32 EPOCHS=5 EXP_NAME=low_lr"},
]

def run_experiment(exp):
    """Reserve GPU, run training, collect results, cancel."""
    timings = {}
    
    # Reserve
    t0 = time.time()
    sb = client.reserve(
        gpu_type="t4",
        gpu_count=1,
        hours=0.5,
        disk_name=exp["disk"],
        name=exp["name"],
    )
    timings['reserve'] = time.time() - t0
    
    # Run training
    t0 = time.time()
    result = sb.exec(f"{exp['env']} python3 /home/dev/train.py", timeout=120)
    timings['train'] = time.time() - t0
    train_output = result.stdout.strip()
    
    # Collect results
    exp_name = exp['env'].split('EXP_NAME=')[1].split()[0]
    result = sb.exec(f"cat /home/dev/results_{exp_name}.json")
    import json
    results = json.loads(result.stdout.strip())
    
    # Cancel
    t0 = time.time()
    sb.cancel()
    timings['cancel'] = time.time() - t0
    
    return {
        'experiment': exp['name'],
        'timings': timings,
        'results': results,
        'train_output': train_output,
    }

# Run both experiments in parallel
t_total = time.time()
with ThreadPoolExecutor(max_workers=2) as pool:
    futures = {pool.submit(run_experiment, exp): exp['name'] for exp in experiments}
    outputs = {}
    for future in as_completed(futures):
        name = futures[future]
        outputs[name] = future.result()
        print(f"✅ {name} completed")

total_time = time.time() - t_total
print(f"\nBoth experiments completed in {total_time:.1f}s (parallel)")

## Step 4: Compare Results

In [ ]:
print("=" * 60)
print(f"{'Metric':<25s} {'High LR':>15s} {'Low LR':>15s}")
print("=" * 60)

high = outputs['high-lr']['results']
low = outputs['low-lr']['results']

print(f"{'Learning Rate':<25s} {high['lr']:>15.4f} {low['lr']:>15.4f}")
print(f"{'Batch Size':<25s} {high['batch_size']:>15d} {low['batch_size']:>15d}")
print(f"{'Final Loss':<25s} {high['final_loss']:>15.4f} {low['final_loss']:>15.4f}")
print(f"{'Avg Epoch Time (s)':<25s} {high['avg_epoch_time']:>15.2f} {low['avg_epoch_time']:>15.2f}")
print()

# Loss progression
print("Loss progression:")
for i in range(len(high['losses'])):
    print(f"  Epoch {i+1}: high_lr={high['losses'][i]:.4f}  low_lr={low['losses'][i]:.4f}")

## Step 5: Timing Breakdown

In [ ]:
print("\n⏱️  Timing Breakdown")
print("=" * 60)
print(f"{'Phase':<25s} {'High LR':>15s} {'Low LR':>15s}")
print("-" * 60)

for phase in ['reserve', 'train', 'cancel']:
    h = outputs['high-lr']['timings'][phase]
    l = outputs['low-lr']['timings'][phase]
    print(f"{phase.capitalize():<25s} {h:>14.1f}s {l:>14.1f}s")

print("-" * 60)
h_total = sum(outputs['high-lr']['timings'].values())
l_total = sum(outputs['low-lr']['timings'].values())
print(f"{'Total (sequential)':<25s} {h_total:>14.1f}s {l_total:>14.1f}s")
print(f"{'Total (parallel)':<25s} {total_time:>14.1f}s {'—':>15s}")
print(f"{'Speedup':<25s} {(h_total + l_total) / total_time:>14.1f}x {'':>15s}")
print()
print(f"Disk clone time: {clone_time:.1f}s")
print(f"Base setup + cancel: {reserve_time + cancel_time:.1f}s")

## Cleanup

Remove the experiment disks if you don't need them.

In [ ]:
# Uncomment to delete experiment disks:
# client.delete_disk("experiment-base")
# client.delete_disk("experiment-variant")
print("Done! Disks preserved for inspection.")
print("Delete with: client.delete_disk('experiment-base')")